In [1]:
# Task 1: Write functions in day12.ipynb, then move them into reader.py, validator.py and transformer.py
from pathlib import Path

READER = r'''"""CSV reading utilities for the Olist dataset."""

import csv
from typing import Any, Dict, List


def read_csv(path: str, delimiter: str = ",") -> List[Dict[str, Any]]:
    """Read a CSV file into a list of dictionaries.

    Args:
        path: Path to the CSV file.
        delimiter: Character separating fields in the file.

    Returns:
        One dictionary per data row, keyed by the header column names.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the file has no header row.
    """
    with open(path, newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle, delimiter=delimiter)
        if reader.fieldnames is None:
            raise ValueError(f"No header row found in {path!r}")
        return [dict(row) for row in reader]
'''

VALIDATOR = r'''"""Lightweight validation helpers shared by the pipeline."""

from typing import Any, Dict, Iterable, List


def validate_required_columns(
    records: List[Dict[str, Any]], required: Iterable[str]
) -> List[str]:
    """Return required column names that are absent from the records.

    Args:
        records: Rows to inspect (a list of dictionaries).
        required: Column names that must be present.

    Returns:
        A list of missing column names, in the order they were requested.
    """
    present = set(records[0]) if records else set()
    return [name for name in required if name not in present]
'''

TRANSFORMER = r'''"""Schema mapping, type conversion and aggregation helpers."""

import re
from datetime import datetime
from typing import Any, Dict, List

TIMESTAMP_COLUMNS = (
    "shipping_limit_date",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_creation_date",
    "review_answer_timestamp",
)

COLUMN_MAPPING: Dict[str, str] = {
    "order_id": "order_id",
    "order_item_id": "order_item_id",
    "product_id": "product_id",
    "seller_id": "seller_id",
    "shipping_limit_date": "shipping_limit_date",
    "price": "price",
    "freight_value": "freight",
}


def to_snake_case(name: str) -> str:
    """Convert a column name to snake_case.

    Args:
        name: Original column name.

    Returns:
        The normalised snake_case name.
    """
    result = re.sub(r"[\-\.\s]+", "_", name.strip())
    result = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", result)
    result = re.sub(r"[^0-9a-zA-Z_]", "", result)
    return re.sub(r"_+", "_", result).strip("_").lower()


def map_columns(
    records: List[Dict[str, Any]], mapping: Dict[str, str]
) -> List[Dict[str, Any]]:
    """Rename columns using a mapping dictionary (unmapped keys are kept).

    Args:
        records: Rows to rename.
        mapping: Source column name -> standard column name.

    Returns:
        A new list of renamed rows.
    """
    return [{mapping.get(key, key): value for key, value in row.items()} for row in records]


def convert_types(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Convert price/freight to float and timestamp fields to datetime.

    Args:
        records: Rows with string values coming from the CSV.

    Returns:
        A new list of rows with converted values.
    """
    result: List[Dict[str, Any]] = []
    for row in records:
        converted = dict(row)
        for field in ("price", "freight"):
            if field in converted and converted[field] not in (None, ""):
                converted[field] = float(converted[field])
        for field in TIMESTAMP_COLUMNS:
            value = converted.get(field)
            if isinstance(value, str) and value:
                converted[field] = datetime.strptime(value, "%Y-%m-%d %H:%M:%S")
        result.append(converted)
    return result


def calculate_order_totals(records: List[Dict[str, Any]]) -> Dict[str, float]:
    """Sum price + freight for every order id using a dictionary.

    Args:
        records: Rows containing order_id, price and freight.

    Returns:
        Mapping of order_id to its total amount.
    """
    totals: Dict[str, float] = {}
    for row in records:
        order_id = row["order_id"]
        totals[order_id] = totals.get(order_id, 0.0) + float(row["price"]) + float(row["freight"])
    return totals
'''

for filename, source in {
    "reader.py": READER,
    "validator.py": VALIDATOR,
    "transformer.py": TRANSFORMER,
}.items():
    Path(filename).write_text(source, encoding="utf-8")

print("Created reader.py, validator.py and transformer.py")

Created reader.py, validator.py and transformer.py


In [2]:
# Task 2: Import your modules into the notebook (use %autoreload)
%load_ext autoreload
%autoreload 2

import reader
import transformer
import validator

print("Imported:", reader.__name__, validator.__name__, transformer.__name__)

Imported: reader validator transformer


In [3]:
# Task 3: Write read_csv(path) returning a list of dictionaries
from pathlib import Path

items_path = Path("../archive/olist_order_items_dataset.csv")
items = reader.read_csv(str(items_path))
print(f"Rows read: {len(items)}")
print("First row:", items[0])

Rows read: 112650
First row: {'order_id': '00010242fe8c5a6d1ba2dd792cb16214', 'order_item_id': '1', 'product_id': '4244733e06e7ecb4970a6e2683c13e61', 'seller_id': '48436dade18ac8b2bce089ec2a041202', 'shipping_limit_date': '2017-09-19 09:45:35', 'price': '58.90', 'freight_value': '13.29'}


In [4]:
# Task 4: Write a function that converts column names to snake_case
samples = ["Freight Value", "orderID", "Product-Category.Name", "  Shipping Limit Date  "]
for sample in samples:
    print(f"{sample!r} -> {transformer.to_snake_case(sample)!r}")

'Freight Value' -> 'freight_value'
'orderID' -> 'order_id'
'Product-Category.Name' -> 'product_category_name'
'  Shipping Limit Date  ' -> 'shipping_limit_date'


In [5]:
# Task 5: Create a column-mapping dictionary to a standard schema
print("Standard schema mapping:")
for source, target in transformer.COLUMN_MAPPING.items():
    print(f"  {source} -> {target}")

standard_rows = transformer.map_columns(items, transformer.COLUMN_MAPPING)
print("Mapped row sample:", standard_rows[0])

Standard schema mapping:
  order_id -> order_id
  order_item_id -> order_item_id
  product_id -> product_id
  seller_id -> seller_id
  shipping_limit_date -> shipping_limit_date
  price -> price
  freight_value -> freight
Mapped row sample: {'order_id': '00010242fe8c5a6d1ba2dd792cb16214', 'order_item_id': '1', 'product_id': '4244733e06e7ecb4970a6e2683c13e61', 'seller_id': '48436dade18ac8b2bce089ec2a041202', 'shipping_limit_date': '2017-09-19 09:45:35', 'price': '58.90', 'freight': '13.29'}


In [6]:
# Task 6: Convert price and freight to float, and timestamps to datetime
typed_rows = transformer.convert_types(standard_rows)
sample = typed_rows[0]
print("price:", sample["price"], type(sample["price"]).__name__)
print("freight:", sample["freight"], type(sample["freight"]).__name__)
print("shipping_limit_date:", sample["shipping_limit_date"], type(sample["shipping_limit_date"]).__name__)

price: 58.9 float
freight: 13.29 float
shipping_limit_date: 2017-09-19 09:45:35 datetime


In [7]:
# Task 7: Calculate order totals per order_id using a dictionary
totals = transformer.calculate_order_totals(typed_rows)
print("Number of orders:", len(totals))
for order_id in list(totals)[:3]:
    print(order_id, round(totals[order_id], 2))

Number of orders: 98666
00010242fe8c5a6d1ba2dd792cb16214 72.19
00018f77f2f0320c557190d7a144bdd3 259.83
000229ec398224ef6ca0657da4fc703e 216.87


In [8]:
# Task 8: Write pytest tests for three functions and run them from the notebook with !pytest
from pathlib import Path

Path("test_pipeline.py").write_text(r'''import pytest

from reader import read_csv
from transformer import calculate_order_totals, to_snake_case


def test_read_csv_returns_list_of_dicts(tmp_path):
    csv_file = tmp_path / "sample.csv"
    csv_file.write_text("a,b\n1,2\n3,4\n", encoding="utf-8")
    rows = read_csv(str(csv_file))
    assert rows == [{"a": "1", "b": "2"}, {"a": "3", "b": "4"}]


def test_to_snake_case():
    assert to_snake_case("Freight Value") == "freight_value"
    assert to_snake_case("orderID") == "order_id"
    assert to_snake_case("Product-Category.Name") == "product_category_name"


def test_calculate_order_totals():
    rows = [
        {"order_id": "A", "price": 10.0, "freight": 2.0},
        {"order_id": "A", "price": 5.0, "freight": 1.0},
        {"order_id": "B", "price": 7.0, "freight": 3.0},
    ]
    assert calculate_order_totals(rows) == {"A": 18.0, "B": 10.0}
''', encoding="utf-8")

import os
import sys

os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get("PATH", "")
!pytest -q

...                                                                      [100%]
3 passed in 0.05s


In [9]:
# Task 9: Add type hints and docstrings; open a PR with passing tests
import inspect

for module in (reader, validator, transformer):
    for name, obj in vars(module).items():
        if inspect.isfunction(obj) and obj.__module__ == module.__name__:
            print(
                f"{module.__name__}.{name}: "
                f"type_hints={bool(obj.__annotations__)}, docstring={bool(obj.__doc__)}"
            )

print("All tests passed above; commit the Day 2 files and open a PR from the pushed branch.")

reader.read_csv: type_hints=True, docstring=True
validator.validate_required_columns: type_hints=True, docstring=True
transformer.to_snake_case: type_hints=True, docstring=True
transformer.map_columns: type_hints=True, docstring=True
transformer.convert_types: type_hints=True, docstring=True
transformer.calculate_order_totals: type_hints=True, docstring=True
All tests passed above; commit the Day 2 files and open a PR from the pushed branch.


In [10]:
# Task 10: Load the column mapping from a JSON config file instead of code
import json
from pathlib import Path

config_path = Path("column_mapping.json")
config_path.write_text(json.dumps(transformer.COLUMN_MAPPING, indent=2), encoding="utf-8")

mapping_from_config = json.loads(config_path.read_text(encoding="utf-8"))
standard_rows_from_config = transformer.map_columns(items, mapping_from_config)
typed_from_config = transformer.convert_types(standard_rows_from_config)
totals_from_config = transformer.calculate_order_totals(typed_from_config)
print("Mapping loaded from config:", mapping_from_config)
print("Totals from config match totals from code:", totals_from_config == totals)

Mapping loaded from config: {'order_id': 'order_id', 'order_item_id': 'order_item_id', 'product_id': 'product_id', 'seller_id': 'seller_id', 'shipping_limit_date': 'shipping_limit_date', 'price': 'price', 'freight_value': 'freight'}
Totals from config match totals from code: True
